# Configuring `KnowledgeGraph` for graph construction

`KnowledgeGraph` is a facade: it owns a build pipeline and a storage index, and
`build_from_docs()` walks documents through both. Everything about *how* the graph
comes out is decided by what you hand the constructor.

```
documents
   │
   ├─ chunker ─────────────────► chunks ──────────────────┐
   │                                                      │
   ├─ artifact_extractor ──────► entities, relations      │
   │                                                      │
   ├─ EntitySummarizer     ┐                              │
   │  RelationSummarizer   ├─ merge duplicates            │
   │                       ┘                              │
   ├─ additional_modules ──────► post-processing          │
   │                                                      │
   ├─ hierarchical Leiden ─────► communities ─► summaries │
   │                                                      │
   └─ Index ◄──────────────────────────────────────────────┘
        graph backend + vector DBs + KV stores
```

Most of this notebook runs **without API keys** — the chunkers, the flag semantics
and the custom modules are all local. Only the "Build it" section near the end
needs `OPENAI_API_KEY`, `LLM_MODEL_NAME` and `EMBEDDER_MODEL_NAME`.

In [ ]:
import dataclasses
import inspect
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    Settings,
    SimpleChunker,
    StorageArguments,
)
from ragu.common.prompts import ICLConfig
from ragu.graph.artifacts_summarizer import EntitySummarizer
from ragu.graph.builder_modules import RemoveIsolatedNodes
from ragu.graph.graph_builder_pipeline import GraphBuilderModule
from ragu.graph.types import Entity, Relation
from ragu.triplet.types import NEREL_ENTITY_TYPES, NEREL_RELATION_TYPES

DATA_DIR = Path("data/ru")

## 1. The constructor

Nine arguments, two of which are objects with their own settings
(`builder_settings`, `storage_settings`).

In [ ]:
for name, param in inspect.signature(KnowledgeGraph.__init__).parameters.items():
    if name == "self":
        continue
    default = "required" if param.default is inspect.Parameter.empty else repr(param.default)
    print(f"{name:20} {default}")

| argument | what it decides |
|---|---|
| `llm` | Extraction, summarization and community summaries. **Optional** — a graph can be built or merely *loaded* without one, but several `BuilderArguments` flags then fail. |
| `embedder` | **Required.** Vectorizes entities, relations and chunks; sizes the vector collections. |
| `sparse_embedder` | Optional BM25/BM42/SPLADE for hybrid retrieval. Needs a Qdrant vector store — see `hybrid_local_search_example.ipynb`. |
| `chunker` | How documents are split. `None` means *one chunk per document*. |
| `artifact_extractor` | How entities and relations are pulled out of a chunk. |
| `builder_settings` | `BuilderArguments` — the subject of most of this notebook. |
| `storage_settings` | `StorageArguments` — where everything lands. See `neo4j_qdrant_setup_example.ipynb`. |
| `additional_modules` | Post-processing hooks that run after extraction. |
| `language` | Overrides `Settings.language` for this graph's prompts. |

### `chunker=None` is a real option

With no chunker, `build_from_docs` treats each document as a single chunk. That is
correct when your parser already produced right-sized units (one PDF page, one
table row, one transcript segment) and wrong when you hand it whole books.

## 2. Global `Settings`

`Settings` is a process-wide singleton. It supplies defaults to everything that was
not given an explicit value, so setting it *before* constructing components matters.

In [ ]:
print(f"language                   {Settings.language}")
print(f"storage_folder             {Settings.storage_folder}")
print(f"llm_context_token_limit    {Settings.llm_context_token_limit}")
print(f"embedder_token_limit       {Settings.embedder_token_limit}")
print(f"tokenizer_llm_name         {Settings.tokenizer_llm_name}")
print(f"tokenizer_llm_backend      {Settings.tokenizer_llm_backend}")
print(f"tokenizer_embedder_name    {Settings.tokenizer_embedder_name}")
print(f"tokenizer_embedder_backend {Settings.tokenizer_embedder_backend}")
print(f"cache_path                 {Settings.cache_path}")
print(f"debug_errors_path          {Settings.debug_errors_path}")

Three of these deserve a note:

- **`storage_folder`** defaults to `./ragu_working_dir/<timestamp>`, so *every run
  writes a new folder*. Set it explicitly to build incrementally into one place —
  otherwise your second run starts from an empty graph and you will think the
  build silently failed.
- **`language`** feeds every prompt and also the BM25 stemmer/stopword list. Set it
  before constructing chunkers, extractors and sparse embedders.
- **`embedder_token_limit`** truncates each text before embedding. Lower it for
  local models with short context windows, or `EmbedderOpenAI` will send inputs the
  server rejects.

`save()` / `load()` persist the annotated fields to JSON. `storage_folder`,
`cache_path` and `debug_errors_path` are deliberately excluded — restoring a
timestamped or machine-specific path silently would send later writes somewhere
stale.

In [ ]:
Settings.language = "russian"
Settings.storage_folder = "ragu_working_dir/kg_settings_example"

# Enabling the response cache makes re-runs of a build nearly free, which is the
# single most useful setting while tuning any of the flags below.
Settings.cache_path = "ragu_working_dir/kg_settings_cache"

## 3. Chunking

The chunker decides what a "unit of evidence" is. Every entity and relation
records the chunk ids it came from, so chunk size directly sets the granularity of
provenance — and the size of the context local search will feed the LLM.

### `SimpleChunker`

Sentence-aware fixed-size splitting. `max_chunk_size` and `overlap` are in
**characters**, not tokens.

In [ ]:
document = (DATA_DIR / "1.txt").read_text(encoding="utf-8")
print(f"document: {len(document)} characters\n")

for size, overlap in [(500, 0), (1000, 0), (1000, 200), (2000, 0)]:
    chunks = SimpleChunker(max_chunk_size=size, overlap=overlap).split(document)
    lengths = [len(chunk.content) for chunk in chunks]
    print(f"max_chunk_size={size:5} overlap={overlap:4} -> {len(chunks):3} chunks, "
          f"mean {sum(lengths) // len(lengths):4} chars, max {max(lengths):4}")

Note that the observed maximum exceeds `max_chunk_size`. That is not a bug:
`SimpleChunker` never splits a sentence, so a chunk that is still under the limit
takes the next whole sentence even if that overshoots. Treat `max_chunk_size` as a
target, not a guarantee — and size your prompts with the *observed* maximum.

Bigger chunks mean fewer extraction calls and more context per call, but coarser
provenance: an entity attributed to a 2000-character chunk points at a whole page
rather than a sentence. Overlap costs duplicated extraction work and buys
resilience against facts that straddle a boundary — note how it also raises the
chunk count, since the overlapped text is re-emitted.

### What a chunk carries

In [ ]:
chunk = SimpleChunker(max_chunk_size=1000).split(document)[0]
for field in dataclasses.fields(chunk):
    value = getattr(chunk, field.name)
    rendered = repr(value)[:60] + "..." if len(repr(value)) > 60 else repr(value)
    print(f"{field.name:18} {rendered}")

`id` is a hash of the content. That is what makes `build_from_docs` idempotent:
re-adding an identical document produces identical chunk ids, and they are
deduplicated before extraction.

### Other chunkers

| chunker | when |
|---|---|
| `SimpleChunker(max_chunk_size, overlap)` | Default. Characters, sentence-aware, no extra dependencies. |
| `SmartSemanticChunker(reranker_name=..., max_chunk_length=250, device=...)` | Splits on semantic boundaries using a reranker. Needs `[local]` and a GPU to be practical. |


## 4. Artifact extraction

The extractor turns a chunk into `(entities, relations)`. Three ship with RAGU.

### `ArtifactsExtractorLLM` — the default

One LLM call per chunk extracts entities and relations together, with optional
validation as a second call.

In [ ]:
for name, param in inspect.signature(ArtifactsExtractorLLM.__init__).parameters.items():
    if name == "self":
        continue
    default = "required" if param.default is inspect.Parameter.empty else repr(param.default)
    print(f"{name:18} {default[:70]}")

- **`do_validation=True`** adds a second LLM call per chunk that re-checks the
  extracted artifacts. Roughly doubles extraction cost; worth it with a weaker
  model, wasteful with a strong one.
- **`entity_types` / `relation_types`** default to the NEREL type lists and are
  injected into the prompt. Pass your own to constrain the ontology, or `None` to
  let the model invent types.

In [ ]:
print(f"NEREL entity types:   {len(NEREL_ENTITY_TYPES)}")
print(f"NEREL relation types: {len(NEREL_RELATION_TYPES)}\n")
for entity_type in NEREL_ENTITY_TYPES[:3]:
    print(f"  {entity_type}")
print("  ...")

A narrower ontology is usually the single biggest quality lever: 29 entity types is
a lot of choice for a small model, and most corpora need six.

In [ ]:
LEGAL_ENTITY_TYPES = [
    "PERSON (a named individual)",
    "ORGANIZATION (a company, agency or institution)",
    "LAW (a statute, act or regulation)",
    "DATE (a calendar date or period)",
    "MONEY (a monetary amount)",
]
LEGAL_RELATION_TYPES = [
    "PARTY_TO (links a person or organization to an agreement)",
    "GOVERNED_BY (links an agreement to the law governing it)",
    "OBLIGATION_OF (links a duty to the party who owes it)",
]

### In-context learning

`ICLConfig` controls the few-shot examples injected into extraction prompts. It is
the cheapest way to stabilize a small model's output format.

In [ ]:
for field in dataclasses.fields(ICLConfig):
    print(f"{field.name:28} {field.default!r}")

`selection_strategy` picks *which* examples get injected for a given chunk:

- `"semantic"` — cosine similarity on dense embeddings. Needs an embedder.
- `"bm25"` — lexical match, no embedder, fast, good for terminology-heavy text.
- `"hybrid"` — reciprocal-rank fusion of the two. Needs an embedder.
- `"random"` — baseline for measuring whether the other three actually help.

`low_match_warning_threshold=0.3` logs a WARNING once 30% of chunks got no examples
at all — the signal that your custom example pool does not match the corpus.

In [ ]:
icl_config = ICLConfig(
    enabled=True,
    num_examples=2,
    selection_strategy="hybrid",
    # examples_base_path=None uses the examples shipped with the package.
)

### The other two extractors

| extractor | shape | when |
|---|---|---|
| `ArtifactsExtractorLLM` | one call per chunk (+1 if validating) | Default. Balanced. |
| `TwoStageArtifactsExtractorLLM` | entities, then relations, each with its own validation toggle | Higher recall on dense text; roughly twice the calls. Separate `do_entity_validation` / `do_relation_validation`. |

## 5. `BuilderArguments`

Ten fields. They fall into three groups: **what to run**, **when it is worth
running**, and **reproducibility**.

In [ ]:
for field in dataclasses.fields(BuilderArguments):
    print(f"{field.name:28} {str(field.default):8} {field.type.__name__}")

### `build_only_vector_context` (default `False`)

The master switch. When `True`, `extract_graph` returns immediately after chunking:
no extraction, no summarization, no communities. You get chunk embeddings and
nothing else — plain vector RAG, and `NaiveSearchEngine` is the only engine that
will find anything.

It also relaxes the requirements: no `artifact_extractor` is needed, and the
summarizers are never constructed.

In [ ]:
vector_only = BuilderArguments(build_only_vector_context=True)
print(f"artifact_extractor required: {not vector_only.build_only_vector_context}")

### `use_llm_summarization` (default `True`)

After extraction, entities are grouped by `(entity_name, entity_type)` and their
descriptions merged. With this on, an LLM rewrites the merged description into one
coherent paragraph; with it off, the descriptions are simply concatenated.

Off is much cheaper and produces visibly repetitive entity descriptions, which then
flow into every retrieval context. On is the default for a reason.

### `summarize_only_if_more_than` (default `7`)

Only entities that were merged from **more than 7 duplicates** are sent to the
summarizing LLM; the rest keep their concatenated description. This is a cost
threshold, and it is the knob to move if summarization dominates your bill.

Lower it (say 2) for a small corpus where few entities repeat often — otherwise
nothing crosses the threshold and `use_llm_summarization=True` quietly does almost
nothing.

In [ ]:
# The summarizer's own rule, verbatim: duplicate_count > summarize_only_if_more_than.
duplicate_counts = [1, 2, 3, 5, 8, 15]

for threshold in (2, 7, 20):
    summarized = [count for count in duplicate_counts if count > threshold]
    print(f"summarize_only_if_more_than={threshold:2} -> LLM call for entities seen "
          f"{summarized if summarized else 'never (nothing crosses the threshold)'}")

### `use_clustering` (default `False`) and `cluster_only_if_more_than` (default `10000`)

This pair is about *one entity with an enormous number of descriptions*. Before
summarizing, the descriptions of a single entity can be DBSCAN-clustered and
summarized cluster-by-cluster, so the summarization prompt stays inside the context
window.

**`cluster_only_if_more_than` counts descriptions of one entity, not entities in
the corpus.** The default of 10 000 means it effectively never triggers — you have
to have an entity appearing in over ten thousand chunks. Lower it to a few hundred
if you have genuine hub entities.

Two traps:

In [ ]:
# Trap 1: clustering without LLM summarization is silently disabled.
summarizer = EntitySummarizer(llm=None, use_llm_summarization=False, use_clustering=True)
print(f"use_clustering after construction: {summarizer.use_clustering}")

Trap 2: clustering requires an embedder. `KnowledgeGraph` always has one, so this
only bites if you build an `EntitySummarizer` yourself.

### `make_community_summary` (default `True`)

Runs hierarchical Leiden over the finished graph, then writes an LLM summary per
detected community. This is what `GlobalSearchEngine` reads — **without it, global
search has nothing to search.**

It is also the most expensive optional stage: one LLM call per community, at every
level of the hierarchy. Turn it off if you only ever use local or naive search.

**Requires an LLM**, and fails at build time rather than construction time.

### `max_cluster_size` (default `128`) and `random_seed` (default `42`)

Passed straight to `hierarchical_leiden`. Smaller `max_cluster_size` yields more,
tighter communities and therefore more summarization calls; larger yields fewer,
broader summaries. `random_seed` makes community detection reproducible — keep it
fixed if you compare builds.

### `remove_isolated_nodes` (default `True`)

Appends a `RemoveIsolatedNodes` module to the pipeline. It drops relations whose
endpoints are missing, then drops entities left with no relations.

Keep it on for graph search: an entity with no edges contributes a description and
no structure, which is exactly what naive chunk search already gives you. Turn it
off if you want the full extracted entity set for inspection.

In [ ]:
entities = [
    Entity(entity_name="Dennis Ritchie", entity_type="PERSON", description="Created C.",
           source_chunk_id=["c1"]),
    Entity(entity_name="Bell Labs", entity_type="ORGANIZATION", description="Research lab.",
           source_chunk_id=["c1"]),
    Entity(entity_name="Orphan", entity_type="PERSON", description="Mentioned once, connected to nothing.",
           source_chunk_id=["c2"]),
]
relations = [
    Relation(subject_id=entities[0].id, object_id=entities[1].id,
             subject_name="Dennis Ritchie", object_name="Bell Labs",
             relation_type="WORKS_AS", description="Worked at Bell Labs.",
             source_chunk_id=["c1"]),
]

kept_entities, kept_relations = await RemoveIsolatedNodes().run(entities, relations)
print(f"{len(entities)} entities -> {len(kept_entities)}: "
      f"{[entity.entity_name for entity in kept_entities]}")

### `vectorize_chunks` (default `False`) — a no-op

Will be removed.

### Dependency summary

| flag | needs `llm` | needs `embedder` | fails when |
|---|---|---|---|
| `use_llm_summarization=True` | yes | no | construction of `KnowledgeGraph` |
| `use_clustering=True` | yes | yes | construction; silently disabled if `use_llm_summarization=False` |
| `make_community_summary=True` | yes | no | `build_from_docs()` |
| `build_only_vector_context=False` | no | no | `build_from_docs()`, if `artifact_extractor` is `None` |

## 6. `additional_modules`

A `GraphBuilderModule` runs after extraction and summarization, before communities
are detected. It receives the full entity and relation lists and must return an
`(entities, relations)` tuple — returning anything else raises `TypeError`.

This is where corpus-specific cleanup belongs: name normalization, dropping noisy
relation types, merging known aliases.

In [ ]:
class DropWeakRelations(GraphBuilderModule):
    """
    Drop relations the extractor was not confident about.

    :param min_strength: Minimum ``relation_strength`` to keep.
    """

    def __init__(self, min_strength: float = 0.5) -> None:
        self.min_strength = min_strength

    async def run(self, entities, relations, **kwargs):
        kept = [r for r in relations if r.relation_strength >= self.min_strength]
        print(f"DropWeakRelations: {len(relations)} -> {len(kept)} relations")
        return entities, kept


weak = Relation(
    subject_id=entities[0].id, object_id=entities[2].id,
    subject_name="Dennis Ritchie", object_name="Orphan",
    relation_type="ACQUAINTANCE_OF", description="Possibly related.",
    relation_strength=0.2, source_chunk_id=["c2"],
)
await DropWeakRelations(min_strength=0.5).run(entities, relations + [weak])

Modules run in the order given. `RemoveIsolatedNodes` is appended *after* your
modules when `remove_isolated_nodes=True`, so a module that deletes relations will
have its orphans cleaned up automatically.

## 7. `storage_settings`

`StorageArguments` decides where the three storage layers live: graph backend,
vector DBs and KV stores. It does not affect *what* is built, only where it lands,
so it is covered end to end in **`neo4j_qdrant_setup_example.ipynb`** and
**`vector_adapters_example.ipynb`**.

In [ ]:
for field in dataclasses.fields(StorageArguments):
    default = field.default_factory() if field.default is dataclasses.MISSING else field.default
    name = getattr(default, "__name__", repr(default))
    print(f"{field.name:28} {name}")

## 8. Build it

From here on you need `OPENAI_API_KEY`, `LLM_MODEL_NAME` and
`EMBEDDER_MODEL_NAME`. This configuration is the "full graph" recipe: LLM
summarization, community summaries, isolated nodes removed.

In [ ]:
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.utils.ragu_utils import read_text_from_files

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

In [ ]:
builder_settings = BuilderArguments(
    use_llm_summarization=True,
    summarize_only_if_more_than=3,   # small corpus: lower than the default 7
    make_community_summary=True,
    remove_isolated_nodes=True,
    use_clustering=False,
    max_cluster_size=128,
    random_seed=42,
)

knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000, overlap=100),
    artifact_extractor=ArtifactsExtractorLLM(
        llm=llm,
        embedder=embedder,
        icl_config=icl_config,
        do_validation=False,
    ),
    builder_settings=builder_settings,
    additional_modules=[DropWeakRelations(min_strength=0.3)],
    language="russian",
)

The expensive cell. With `Settings.cache_path` set above, a second run of the same
configuration replays from cache instead of paying for it again.

In [ ]:
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

In [ ]:
entities = await knowledge_graph.index.graph_backend.get_all_nodes()
relations = await knowledge_graph.index.graph_backend.get_all_edges()
communities = await knowledge_graph.index.community_kv_storage.all_keys()

print(f"entities:    {len(entities)}")
print(f"relations:   {len(relations)}")
print(f"communities: {len(communities)}")
print(f"chunks:      {len(await knowledge_graph.index.chunks_kv_storage.all_keys())}")

## 9. Incremental builds

`build_from_docs` can be called repeatedly on the same graph. Two mechanisms make
that safe:

1. **Chunk ids are content hashes**, and duplicates are dropped before extraction.
   Re-adding an unchanged document costs nothing and logs a warning naming the
   duplicate count.
2. **Cluster ids are remapped** to stay globally unique per level across runs, so
   the communities from run 2 do not collide with those from run 1.

What is *not* automatic: community structure is only computed over the artifacts of
the current call. After merging new documents into an existing graph, the old
communities are stale.

In [ ]:
before = len(await knowledge_graph.index.chunks_kv_storage.all_keys())
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))
after = len(await knowledge_graph.index.chunks_kv_storage.all_keys())

print(f"chunks before: {before}, after re-adding the same corpus: {after}")

### Reindexing

| method | does |
|---|---|
| `reindex_descriptions(summarize_only_more_than=None)` | Re-summarizes entity/relation descriptions longer than N sentences. Requires `use_llm_summarization=True`. |
| `reindex_community()` | Re-runs Leiden over the whole stored graph and regenerates every community summary. Drops the old community stores first. |
| `reindex_graph()` | Both, in that order. The thing to call after merging new documents in. |

In [ ]:
await knowledge_graph.reindex_graph()

print(f"communities after reindex: "
      f"{len(await knowledge_graph.index.community_kv_storage.all_keys())}")

## 10. Recipes

### Naive vector RAG — no graph at all

```python
KnowledgeGraph(
    llm=None,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    builder_settings=BuilderArguments(build_only_vector_context=True),
)
```
No extractor needed, no LLM needed. Only `NaiveSearchEngine` will work.

### Cheap graph — structure without polish

```python
BuilderArguments(
    use_llm_summarization=False,   # concatenate descriptions instead of rewriting
    make_community_summary=False,  # no global search
    remove_isolated_nodes=True,
)
```
Extraction is the only LLM cost. Local search works; global search does not.

### Full graph — everything on

```python
BuilderArguments(
    use_llm_summarization=True,
    summarize_only_if_more_than=3,
    make_community_summary=True,
    remove_isolated_nodes=True,
)
```

### Large corpus with hub entities

```python
BuilderArguments(
    use_llm_summarization=True,
    use_clustering=True,
    cluster_only_if_more_than=300,  # the default 10000 effectively never fires
    max_cluster_size=64,            # more, tighter communities
)
```

### Cheapest extraction at scale

```python
KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1500),
    artifact_extractor=RaguLmArtifactExtractor(llm=ragu_lm),
    builder_settings=BuilderArguments(make_community_summary=False),
)
```


In [ ]:
await knowledge_graph.index.close()